# 06 — Gold Rebuild (M2)

Rebuilds **only the Gold facts affected** by the corrected Silver (brief §8.3 — partial rebuilds are acceptable; do NOT blindly rebuild all 27 tables). Reuses M1's Lead-fact build logic from `04_gold_facts.ipynb`, **re-pointed at corrected Silver** with header-level corrections propagated into the line/measure grain. Each write uses `overwrite=True` (idempotent).

## Affected → rebuild map (only these)
| Corrected Silver | Anomalies | Gold to rebuild |
|---|---|---|
| silver_ec_orders / order_lines | A1,A2,A3,A11,B1 | **fact_ecommerce_order_line** (keystone) |
| silver_pos / returns | A1,A3 | fact_store_sale_line, fact_return_line *(M5 build logic absent — see S2)* |
| silver_wh / si inventory + movements + receipts | A5,A6,A7,A8,A10,B3 | fact_warehouse_inventory_snapshot, fact_store_inventory_snapshot, fact_inventory_transaction |
| silver_dc_delivery_events | A14,B7 | fact_order_fulfilment |
| silver_product_master | A9,B4 | dim_product *(descriptive — left intact, FK resolves; see S2)* |
| silver_customer_master | A11,B5 | dim_customer *(descriptive — left intact; see S2)* |
| silver_nl_listings/events | A4,A12,A13,B6 | fact_classified_listing_event, fact_classified_listing_snapshot |
| silver_rv_reviews | A15 | fact_customer_review |
| silver_cs_cases | A16 | fact_customer_complaint |
| silver_ts_sellers | A13,B8 | dim_seller, fact_seller_performance_snapshot *(descriptive/secondary — see S2)* |

**Rebuild-exempt (static lookups, no Silver correction feeds them):** dim_date, dim_channel, dim_step, dim_payment_method, dim_delivery_method, dim_listing_condition, dim_return_reason, dim_promotion.

**Scope note (honest):** the affected **facts** are rebuilt from corrected Silver below. Descriptive **dims** (dim_customer/product/seller) are left intact — facts carry their FK as `surrogate_key(natural_code)`, so joins still resolve, and a full SCD2 dim rebuild is the member task. fact_store_sale_line / fact_return_line / fact_seller_performance_snapshot have no M1 build logic in this repo (member TODO) and are noted in S2 rather than rebuilt blind.

**Rebuild order:** dims (unchanged) → facts.

In [ ]:
%pip install -q snowflake-connector-python pandas
# NOTE: dbutils.library.restartPython() intentionally omitted — it hangs on Free Edition serverless.

In [ ]:
import os, sys, glob
def _add_shared():
    cands = (glob.glob('/Workspace/Users/*/nexamart-m2/notebooks/_shared')
             + glob.glob('/Workspace/Users/*/nexamart_m2/notebooks/_shared')
             + [os.path.join(os.getcwd(), 'notebooks', '_shared'),
                os.path.join(os.getcwd(), '_shared')])
    for c in cands:
        if os.path.isdir(c) and c not in sys.path:
            sys.path.append(c)
            return c
    raise RuntimeError(f'_shared not found; tried: {cands}')
print('shared dir:', _add_shared())

dbutils.widgets.text('sf_account', 'rhxendw-yb24678')
dbutils.widgets.text('sf_user', 'NEXAMART_LEAD')
dbutils.widgets.text('sf_password', '')
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role', 'NEXAMART_ENGINEER')

from pyspark.sql import functions as F, Window
import utils_snowflake as sf
from utils_keys import surrogate_key, anonymous_key
from utils_dates import parse_date
from utils_anomaly import add_anomaly_columns, flag

def read_bronze(t): return sf.read_from_snowflake(spark, t, schema='NEXAMART_BRONZE')
def read_silver(t): return sf.read_from_snowflake(spark, t, schema='NEXAMART_SILVER')
def read_gold(t):   return sf.read_from_snowflake(spark, t, schema='NEXAMART_GOLD')
def write_gold(df, t):
    n = sf.write_to_snowflake(df, t, schema='NEXAMART_GOLD', overwrite=True)
    print(f'Wrote gold.{t} ({n} rows)'); return n
def date_key(col): return surrogate_key(col)   # matches dim_date (surrogate over the ISO date string)
def assert_grain(df, keys, name):
    total = df.count(); distinct = df.select(*keys).distinct().count()
    ok = (total == distinct)
    print(f"[grain] {name}: rows={total}  distinct({', '.join(keys)})={distinct}  -> {'PASS' if ok else 'FAIL'}")
    assert ok, f'GRAIN VIOLATION in {name}'
    return total

# Facts rebuilt here (those an affected Silver entity feeds). Descriptive dims are left intact —
# the facts carry member-dim FKs as surrogate_key(natural_code), so the joins still resolve; a full
# SCD2 dim rebuild is a member task (documented in report S2).
AFFECTED = ['fact_ecommerce_order_line', 'fact_order_fulfilment', 'fact_store_inventory_snapshot',
            'fact_warehouse_inventory_snapshot', 'fact_inventory_transaction',
            'fact_classified_listing_snapshot', 'fact_classified_listing_event',
            'fact_customer_review', 'fact_customer_complaint']
before = {}
for t in AFFECTED:
    try:
        before[t] = read_gold(t).count()
    except Exception as e:
        before[t] = None
        print(f'before-count skipped for {t}:', repr(e))
print('BEFORE counts:', before)

## Step 1 — rebuild affected DIMENSIONS first

In [ ]:
# Descriptive dims (dim_customer/product/seller) are intentionally NOT rebuilt here.
# Reasons: (1) facts carry their dim FK as surrogate_key(natural_code) — the same convention the
# SCD2 dims use — so every fact->dim join still resolves after the fact rebuild; (2) the SCD2 history
# build for those member dims is not in this repo (member-owned), and a blind overwrite would replace
# the live SCD2 schema. The descriptive corrections (A9 canonical product, B5 merged identity, B8 risk
# tier) are low-impact on the financial KPIs and are documented in report S2 as a member follow-up.
# Static lookup dims (dim_date/channel/step/payment_method/delivery_method/...) have no Silver feed.
print('Dimensions left intact (FKs resolve via surrogate_key(natural_code)); rebuilding affected FACTS only.')

## Step 2 — rebuild affected FACTS (after their dims)

In [ ]:
# === fact_ecommerce_order_line (KEYSTONE) — rebuild from corrected silver_ec_orders ===
# Grain: (order_id, line_id). Reuses 04's logic, re-pointed at CORRECTED Silver. Corrections
# propagated: A1 -> zero line_subtotal_excl_tax where the order is CANCELLED (header fix flows to the
# line measure); A11 -> corrected customer_id already in silver_ec_orders feeds customer_key;
# B1 -> attribution via the bridge (102) unchanged.
lines  = read_silver('silver_ec_order_lines')
orders = read_silver('silver_ec_orders').select(
    'order_id', 'order_date', 'customer_id', 'delivery_method_code', 'pickup_store_id',
    'promo_code_applied', 'order_status')
bridge = read_silver('silver_campaign_attribution_bridge').select(
    'order_id', F.lit(True).alias('is_bridge'), 'attribution_rule').dropDuplicates(['order_id'])

f = lines.join(orders, 'order_id', 'left').join(bridge, 'order_id', 'left')
f = (f
     .withColumn('date_key',            date_key(F.col('order_date')))
     .withColumn('product_key',         surrogate_key(F.col('product_sku')))
     .withColumn('customer_key',        surrogate_key(F.col('customer_id')))            # A11 corrected
     .withColumn('channel_key',         surrogate_key(F.when(F.col('delivery_method_code') == 'BOPIS', F.lit('BOPIS')).otherwise(F.lit('EC'))))
     .withColumn('delivery_method_key', surrogate_key(F.col('delivery_method_code')))
     .withColumn('promotion_key',       surrogate_key(F.col('promo_code_applied')))
     .withColumn('store_key',           surrogate_key(F.col('pickup_store_id')))
     .withColumn('quantity',            F.col('quantity').cast('int'))
     # A1 propagation: cancelled orders contribute zero line revenue
     .withColumn('line_subtotal_excl_tax',
                 F.when(F.col('order_status') == 'CANCELLED', F.lit(0.0))
                  .otherwise(F.col('line_total_excl_tax').cast('double')))
     .withColumn('discount_amount',     F.col('discount_applied').cast('double'))
     .withColumn('unit_price_excl_tax', F.col('unit_price_excl_tax').cast('double'))
     .withColumn('is_campaign_attributed', F.coalesce(F.col('is_bridge'), F.lit(False)))
     .withColumn('attribution_rule',
                 F.when(F.col('promo_code_applied').isNotNull(), F.lit('PROMO_CODE'))
                  .when(F.col('is_bridge') == True, F.lit('SESSION_BRIDGE')).otherwise(F.lit(None))))
f = add_anomaly_columns(f)
f = flag(f, F.col('is_campaign_attributed') == True, 'ATTRIBUTION_SESSION_BRIDGE', 'CLEAN', 'INFERRED')
f = flag(f, F.col('order_status') == 'CANCELLED', 'CANCELLED_WITH_REVENUE', 'FLAGGED_ANOMALY', 'UNRELIABLE')

fact_ecol = f.select(
    'order_id', 'line_id', 'date_key', 'product_key', 'customer_key', 'channel_key',
    'delivery_method_key', 'promotion_key', 'store_key', 'product_sku', 'customer_id',
    'quantity', 'line_subtotal_excl_tax', 'discount_amount', 'unit_price_excl_tax',
    'is_campaign_attributed', 'attribution_rule',
    'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level')
write_gold(fact_ecol, 'fact_ecommerce_order_line')
assert_grain(fact_ecol, ['order_id', 'line_id'], 'fact_ecommerce_order_line')

In [ ]:
# === fact_order_fulfilment (A14, B7) — accumulating snapshot, grain = order_id ===
# Reuses 04 logic; A14 propagated by using corrected_event_datetime where set (COALESCE with
# event_ts_parsed) for the PICKED_UP / DELIVERED milestones.
orders = read_silver('silver_ec_orders').select('order_id', 'order_number', 'order_date', 'order_status', 'customer_id')
pg = (read_bronze('pg_transactions').filter(F.col('captured_ts').isNotNull())
      .groupBy('order_ref').agg(F.min(F.col('captured_ts').cast('long')).alias('captured_epoch')))
ship = read_bronze('dc_shipments').select('shipment_id', F.col('order_reference').alias('order_number'))
ev = (read_silver('silver_dc_delivery_events')
      .select('shipment_id', 'event_type_code',
              F.coalesce(F.col('corrected_event_datetime'), F.col('event_ts_parsed')).alias('eff_ts'),
              'anomaly_reason_code'))
ev2 = ev.join(ship, 'shipment_id', 'inner')
picked    = ev2.filter(F.col('event_type_code') == 'PICKED_UP').groupBy('order_number').agg(F.min('eff_ts').alias('picked_ts'))
delivered = ev2.filter(F.col('event_type_code') == 'DELIVERED').groupBy('order_number').agg(F.min('eff_ts').alias('delivered_ts'))
drift     = (ev2.filter(F.col('anomaly_reason_code').contains('DELIVERY_BEFORE_SHIP'))
                .select('order_number').distinct().withColumn('a14', F.lit(True)))
ret = (read_bronze('rr_return_requests')
       .withColumn('returned_dt', parse_date(F.col('return_request_date'), 'ddmonyyyy'))
       .groupBy(F.col('original_order_ref').alias('order_number')).agg(F.min('returned_dt').alias('returned_dt')))
f = (orders
     .join(pg, orders.order_number == pg.order_ref, 'left').drop('order_ref')
     .join(picked, 'order_number', 'left').join(delivered, 'order_number', 'left')
     .join(ret, 'order_number', 'left').join(drift, 'order_number', 'left'))
dk = lambda ts: F.when(ts.isNotNull(), date_key(F.date_format(ts, 'yyyy-MM-dd')))
f = (f
     .withColumn('placed_ts',   parse_date(F.col('order_date'), 'iso_date').cast('timestamp'))
     .withColumn('captured_ts', F.col('captured_epoch').cast('timestamp'))
     .withColumn('placed_date_key',    date_key(F.col('order_date')))
     .withColumn('captured_date_key',  dk(F.col('captured_ts')))
     .withColumn('shipped_date_key',   dk(F.col('picked_ts')))
     .withColumn('delivered_date_key', dk(F.col('delivered_ts')))
     .withColumn('returned_date_key',  dk(F.col('returned_dt').cast('timestamp')))
     .withColumn('placed_to_captured_hours',  (F.col('captured_ts').cast('long')  - F.col('placed_ts').cast('long')) / 3600.0)
     .withColumn('picked_to_delivered_hours', (F.col('delivered_ts').cast('long') - F.col('picked_ts').cast('long')) / 3600.0)
     .withColumn('total_lifecycle_hours',     (F.col('delivered_ts').cast('long') - F.col('placed_ts').cast('long')) / 3600.0)
     .withColumn('customer_key', surrogate_key(F.col('customer_id'))))
f = add_anomaly_columns(f)
f = flag(f, F.col('a14') == True, 'DELIVERY_BEFORE_SHIP', 'FLAGGED_ANOMALY', 'UNRELIABLE')
fact_ful = f.select(
    'order_id', 'order_number', 'customer_key', 'customer_id', 'order_status',
    'placed_date_key', 'captured_date_key', 'shipped_date_key', 'delivered_date_key', 'returned_date_key',
    'placed_to_captured_hours', 'picked_to_delivered_hours', 'total_lifecycle_hours',
    'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level')
write_gold(fact_ful, 'fact_order_fulfilment')
assert_grain(fact_ful, ['order_id'], 'fact_order_fulfilment')


# --- snapshot_date may be a DATE (corrected Silver) or DD/MM/YYYY text — normalise to ISO ---
def _iso_date(col):
    return F.coalesce(F.date_format(F.to_date(col), 'yyyy-MM-dd'),
                      F.date_format(parse_date(col, 'ddmmyyyy'), 'yyyy-MM-dd'))

# === fact_store_inventory_snapshot (A6, A8) — grain store_id x product_code x snapshot_date ===
# Re-point the clean rows to CORRECTED silver_si_inventory_snapshots (A6 negative->0) + UNION the
# A8 reconstructed sibling. Guarded: member Silver column set is less certain than the verified core.
try:
    si_s = read_silver('silver_si_inventory_snapshots')
    _sic = set(si_s.columns)
    _pcol = 'product_code' if 'product_code' in _sic else ('sku' if 'sku' in _sic else None)
    clean = (si_s
             .withColumn('reserved_qty', F.col('reserved_qty').cast('int') if 'reserved_qty' in _sic else F.lit(None).cast('int'))
             .withColumn('damaged_qty',  F.col('damaged_qty').cast('int')  if 'damaged_qty'  in _sic else F.lit(None).cast('int'))
             .select(F.col('store_id').cast('string').alias('store_id'),
                     F.col(_pcol).alias('product_code'),
                     _iso_date(F.col('snapshot_date')).alias('snapshot_iso'),
                     F.col('physical_qty').cast('int').alias('physical_qty'),
                     F.col('sellable_qty').cast('int').alias('sellable_qty'),
                     'reserved_qty', 'damaged_qty')
             .withColumn('is_reconstructed', F.lit(False)))
    clean = add_anomaly_columns(clean)
    recon = read_silver('silver_store_inventory_snapshots_reconstructed')
    _rc = set(recon.columns)
    recon = recon.select(
        F.col('store_id').cast('string').alias('store_id'), F.col('product_code').alias('product_code'),
        _iso_date(F.col('snapshot_date')).alias('snapshot_iso'),
        F.col('physical_qty').cast('int').alias('physical_qty'), F.col('sellable_qty').cast('int').alias('sellable_qty'),
        F.lit(None).cast('int').alias('reserved_qty'), F.lit(None).cast('int').alias('damaged_qty'),
        F.lit(True).alias('is_reconstructed'),
        'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level')
    both = clean.unionByName(recon)
    fact_sis = (both
                .withColumn('date_key',    date_key(F.col('snapshot_iso')))
                .withColumn('product_key', surrogate_key(F.col('product_code')))
                .withColumn('store_key',   surrogate_key(F.col('store_id'))))
    write_gold(fact_sis, 'fact_store_inventory_snapshot')
    assert_grain(fact_sis, ['store_id', 'product_code', 'snapshot_iso'], 'fact_store_inventory_snapshot')
except Exception as e:
    print('fact_store_inventory_snapshot rebuild needs a runtime column fix:', repr(e))

# === fact_warehouse_inventory_snapshot (A5) — grain warehouse_id x sku x snapshot_date ===
# Member fact (no M1 logic in repo) — authored from the 04 spec, reading CORRECTED Silver (A5 ATP->0).
try:
    wh_s = read_silver('silver_wh_inventory_snapshots')
    _whc = set(wh_s.columns)
    fact_whs = (wh_s
                .withColumn('snapshot_iso', _iso_date(F.col('snapshot_date')))
                .withColumn('date_key',      date_key(F.col('snapshot_iso')))
                .withColumn('product_key',   surrogate_key(F.col('sku')))
                .withColumn('warehouse_key', surrogate_key(F.col('warehouse_id').cast('string')))
                .select(F.col('warehouse_id').cast('string').alias('warehouse_id'), 'sku', 'snapshot_iso',
                        'date_key', 'product_key', 'warehouse_key',
                        F.col('physical_qty').cast('int').alias('physical_qty'),
                        F.col('atp_qty').cast('int').alias('atp_qty'),
                        'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level'))
    write_gold(fact_whs, 'fact_warehouse_inventory_snapshot')
    assert_grain(fact_whs, ['warehouse_id', 'sku', 'snapshot_iso'], 'fact_warehouse_inventory_snapshot')
except Exception as e:
    print('fact_warehouse_inventory_snapshot rebuild needs a runtime column fix:', repr(e))

# === fact_inventory_transaction (B3) — grain movement_id x node_type ===
# si movements from Bronze (unaffected); WH movements from CORRECTED Silver (B3 classification).
try:
    si_m = (read_bronze('si_inventory_movements')
            .select('movement_id', F.lit('STORE').alias('node_type'),
                    F.col('store_id').cast('string').alias('node_id'), F.col('product_code').alias('product_code'),
                    'movement_type', F.col('quantity_change').cast('int').alias('quantity_delta'),
                    'reference_number', parse_date(F.col('movement_date'), 'ddmmyyyy').alias('mdate')))
    wh_m0 = read_silver('silver_wh_inventory_movements')
    _wmc = set(wh_m0.columns)
    _wm_dt = 'movement_dt_parsed' if 'movement_dt_parsed' in _wmc else None
    wh_m = (wh_m0
            .select('movement_id', F.lit('WAREHOUSE').alias('node_type'),
                    F.col('warehouse_id').cast('string').alias('node_id'), F.col('sku').alias('product_code'),
                    'movement_type', F.col('quantity_change').cast('int').alias('quantity_delta'),
                    'reference_number',
                    (F.col(_wm_dt) if _wm_dt else F.to_date(F.col('movement_datetime'))).alias('mdate')))
    mv = si_m.unionByName(wh_m)
    fact_it = (add_anomaly_columns(mv)
               .withColumn('date_key',    date_key(F.date_format(F.col('mdate'), 'yyyy-MM-dd')))
               .withColumn('product_key', surrogate_key(F.col('product_code')))
               .withColumn('node_key',    surrogate_key(F.col('node_type'), F.col('node_id'))))
    fact_it = flag(fact_it, F.col('reference_number').isNull() & (F.col('node_type') == 'WAREHOUSE'),
                   'MOVEMENT_NULL_REF', 'FLAGGED_AMBIGUOUS', 'INFERRED')
    write_gold(fact_it, 'fact_inventory_transaction')
    assert_grain(fact_it, ['movement_id', 'node_type'], 'fact_inventory_transaction')
except Exception as e:
    print('fact_inventory_transaction rebuild needs a runtime column fix:', repr(e))

In [ ]:
# === fact_classified_listing_snapshot (A12, A13) — grain listing_id ===
# Reuses 04 logic against CORRECTED silver_nl_listings; carries the A12/A13 flags. Defensive on the
# optional descriptive columns (member Silver naming varies; created_at has no _parsed suffix).
try:
    nl = read_silver('silver_nl_listings'); _nlc = set(nl.columns)
    _opt = lambda c, t='string': (F.col(c) if c in _nlc else F.lit(None).cast(t))
    _created = next((c for c in ('created_at_parsed', 'created_at') if c in _nlc), None)
    dkey = (date_key(F.date_format(F.to_timestamp(F.col(_created)), 'yyyy-MM-dd')) if _created else F.lit(None).cast('string'))
    fact = (nl
            .withColumn('date_key',    dkey)
            .withColumn('product_key', surrogate_key(_opt('category_code')))
            .withColumn('seller_key',  surrogate_key(F.col('seller_account_id')))
            .select('listing_id', _opt('listing_ref').alias('listing_ref'),
                    'date_key', 'product_key', 'seller_key', 'seller_account_id',
                    _opt('category_code').alias('category_code'), 'status_code',
                    _opt('asking_price', 'double').cast('double').alias('asking_price'),
                    _opt('views_count', 'int').cast('int').alias('views_count'),
                    _opt('favourites_count', 'int').cast('int').alias('favourites_count'),
                    _opt('relist_count', 'int').cast('int').alias('relist_count'),
                    'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level'))
    write_gold(fact, 'fact_classified_listing_snapshot')
    assert_grain(fact, ['listing_id'], 'fact_classified_listing_snapshot')
except Exception as e:
    print('fact_classified_listing_snapshot rebuild needs a runtime column fix:', repr(e))

# === fact_classified_listing_event (A4, B6) — grain (listing_id, event_id) ===
# Member fact (no M1 logic in repo) — from the 04 spec, CORRECTED silver_nl_listing_events. SELLER_SOLD
# stays ESTIMATED (A4); the 4 signal types carry ESTIMATED_NL_GMV (B6).
try:
    nle = read_silver('silver_nl_listing_events'); _ec = set(nle.columns)
    _opt = lambda c, t='string': (F.col(c) if c in _ec else F.lit(None).cast(t))
    _ev_ts = next((c for c in ('event_ts_parsed', 'event_timestamp', 'event_datetime') if c in _ec), None)
    _ev_id = next((c for c in ('event_id', 'event_seq') if c in _ec), None)
    dkey = (date_key(F.date_format(F.to_timestamp(F.col(_ev_ts)), 'yyyy-MM-dd')) if _ev_ts else F.lit(None).cast('string'))
    fact = (nle
            .withColumn('event_id_dg', F.col(_ev_id).cast('string') if _ev_id else F.monotonically_increasing_id().cast('string'))
            .withColumn('date_key',    dkey)
            .withColumn('product_key', surrogate_key(_opt('listing_id')))
            .select('listing_id', 'event_id_dg', 'date_key', 'product_key',
                    'event_type_code', _opt('offer_amount', 'double').cast('double').alias('offer_amount'),
                    'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level'))
    write_gold(fact, 'fact_classified_listing_event')
    assert_grain(fact, ['listing_id', 'event_id_dg'], 'fact_classified_listing_event')
except Exception as e:
    print('fact_classified_listing_event rebuild needs a runtime column fix:', repr(e))

# === fact_customer_review (A15) — grain review_id ===
# Member fact (no M1 logic in repo) — from the 04 spec, CORRECTED silver_rv_reviews (A15 set
# is_verified_purchase=FALSE for pre-delivery reviews).
try:
    rv = read_silver('silver_rv_reviews'); _rc = set(rv.columns)
    _opt = lambda c, t='string': (F.col(c) if c in _rc else F.lit(None).cast(t))
    _rv_ts = next((c for c in ('review_ts_parsed', 'review_datetime', 'review_date') if c in _rc), None)
    dkey = (date_key(F.date_format(F.to_timestamp(F.col(_rv_ts)), 'yyyy-MM-dd')) if _rv_ts else F.lit(None).cast('string'))
    fact = (rv
            .withColumn('date_key',     dkey)
            .withColumn('product_key',  surrogate_key(_opt('product_sku')))
            .withColumn('customer_key', surrogate_key(_opt('customer_id')))
            .select('review_id', 'date_key', 'product_key', 'customer_key',
                    _opt('rating', 'int').cast('int').alias('star_rating'),
                    _opt('is_verified_purchase', 'int').cast('int').alias('is_verified_purchase'),
                    _opt('days_post_delivery', 'int').cast('int').alias('days_post_delivery'),
                    'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level'))
    write_gold(fact, 'fact_customer_review')
    assert_grain(fact, ['review_id'], 'fact_customer_review')
except Exception as e:
    print('fact_customer_review rebuild needs a runtime column fix:', repr(e))

# === fact_customer_complaint (A16) — grain case_id ===
# Reuses 04 logic re-pointed to CORRECTED silver_cs_cases (canonical_case_key + A16 resolution).
try:
    cs = read_silver('silver_cs_cases'); _cc = set(cs.columns)
    _opt = lambda c, t='string': (F.col(c) if c in _cc else F.lit(None).cast(t))
    _canon = next((c for c in ('canonical_case_key', 'canonical_case_ref') if c in _cc), None)
    _ref = next((c for c in ('case_reference', 'case_ref') if c in _cc), None)
    _cust = next((c for c in ('customer_ref', 'customer_id') if c in _cc), None)
    _open_ts = next((c for c in ('created_dt_parsed', 'case_open_datetime', 'created_datetime') if c in _cc), None)
    dkey = (date_key(F.date_format(F.to_timestamp(F.col(_open_ts)), 'yyyy-MM-dd')) if _open_ts else F.lit(None).cast('string'))
    fact = (cs
            .withColumn('canonical_case_key', F.coalesce(F.col(_canon), F.col(_ref)) if (_canon and _ref) else (F.col(_canon) if _canon else _opt('case_id')))
            .withColumn('date_key',     dkey)
            .withColumn('customer_key', surrogate_key(_opt(_cust)))
            .withColumn('channel_key',  surrogate_key(_opt('channel_attribution')))
            .select('case_id', 'canonical_case_key', _opt(_ref).alias('case_reference'),
                    'date_key', 'customer_key', 'channel_key',
                    _opt('is_duplicate_flag').alias('is_duplicate_flag'),
                    'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level'))
    write_gold(fact, 'fact_customer_complaint')
    assert_grain(fact, ['case_id'], 'fact_customer_complaint')
except Exception as e:
    print('fact_customer_complaint rebuild needs a runtime column fix:', repr(e))

## Step 3 — grain assertions + before/after delta (for report S2)

In [ ]:
# Grain asserts already ran inline per fact (assert_grain). Here: before/after row-count delta for S2.
# Corrections change MEASURES / flags, not grain — so row counts should be IDENTICAL before vs after;
# a non-zero delta means the rebuild dropped/added rows and must be investigated.
after = {}
for t in AFFECTED:
    try:
        after[t] = read_gold(t).count()
    except Exception as e:
        after[t] = None
print('Gold rebuild delta (report S2):')
print(f"{'table':42s} {'before':>10s} {'after':>10s} {'delta':>8s}")
for t in AFFECTED:
    b, a = before.get(t), after.get(t)
    d = (a - b) if (isinstance(a, int) and isinstance(b, int)) else None
    flag_ = '' if (d == 0 or d is None) else '  <-- ROW COUNT CHANGED'
    print(f"{t:42s} {str(b):>10s} {str(a):>10s} {str(d):>8s}{flag_}")
print('\nNext: run sql/validation_suite.sql in a Snowflake worksheet — all 10 checks should pass.')